In [ ]:
# imports

In [6]:
import sys
sys.path.append('../../../common_code')

In [7]:
import sqlite3
from paths import PATH_ROOT
from common_code.db_operations.verb_transactions.transactions_filtering import *

Transaktsioonide filtreerimine kasutades funktsioone *create_filtered_transaction_row* ja *create_filtered_transaction_head*.

In [8]:
# ühenduse loomine verbimustrite andmebaasiga
con = sqlite3.connect("C:/Users/liivas/Documents/Töö/verbirektisoonid/vp_data3.db")
cur = con.cursor()

In [9]:
# transaktsioonide andmebaasi lisamine (v32)
cur.execute('ATTACH DATABASE "C:/Users/liivas/Documents/Töö/verbirektisoonid/v32_data.db" AS v32')

In [10]:
# uue andmebaasi lisamine transaktsioonide jaoks, kust on eemaldatud olemasolevad verbimustrid
cur.execute('ATTACH DATABASE "v32_data_filtered.db" AS filtered')

In [11]:
cur.execute('ATTACH DATABASE "C:/Users/liivas/Documents/Töö/verbirektisoonid/db_operations/verb_negations/negations.db" AS neg')

In [12]:
# uue andmebaasi lisamine filtreeritud transaktsioonide jaoks, kust on omakorda eemaldatud eitused
cur.execute('ATTACH DATABASE "v32_data_filtered_neg.db" AS filtered_neg')

In [13]:
# uue andmebaasi lisamine filtreeritud transaktsioonide jaoks, kust on omakorda eemaldatud eitused
cur.execute('ATTACH DATABASE "v32_data_filtered_no_neg.db" AS filtered_no_neg')

### Filtreeritud transaktsioonid

In [8]:
# transaktsioonide ID-de tabeli loomine, mis vastavad transaktsioonidele, mis ei vasta olemasolevatele mustritele

create_filtered_head_id_tbl(cur,
                           all_ids_tbl='verb_matches', # andmebaasist vp_data3 mustritele vastavad verbid
                           head_id_col1='head_id',
                           ids_to_filter_tbl='verb_phrase_matches', # andmebaasist vp_data3 tervikmustreid sisaldavad fraasid
                           head_id_col2='head_id',
                           output_tbl='filtered.filtered_head_ids')

In [9]:
# filtreeritud transaction_row
create_filtered_transaction_row(cur,
                                head_ids='filtered.filtered_head_ids',
                                head_id_col='head_id'
                                transaction_row='v32.transaction_row',
                                output_tr_row='filtered.transaction_row'
                               )

In [10]:
# filtreeritud transaction_head
create_filtered_transaction_head(cur,
                                 head_ids='filtered.filtered_head_ids',
                                 head_id_col='head_id'
                                 transaction_head='v32.transaction_head',
                                 output_tr_head='filtered.transaction_head')

In [11]:
# transaction_row täiendav 'deprel' tulba väärtuste filtreerimine

remove_deprel_from_transaction_row(cur,
                                   transaction_row='filtered.transaction_row',
                                   deprel='nsubj')
remove_deprel_from_transaction_row(cur,
                                   transaction_row='filtered.transaction_row',
                                   deprel='advmod')
remove_deprel_from_transaction_row(cur,
                                   transaction_row='filtered.transaction_row',
                                   deprel='advcl')
remove_deprel_from_transaction_row(cur,
                                   transaction_row='filtered.transaction_row',
                                   deprel='csubj')

In [12]:
# transaction_row hulgast abiverbidest AUX-de eemaldamine

remove_aux_verbs(cur,
                 transaction_row='filtered.transaction_row')

### Filtreeritud transaktsioonid ainult eitustega

In [14]:
# filtreeritud transaction_row
create_filtered_transaction_row(cur,
                                'neg.neg_phrase_matches', # andmebaasist neg_tables mustritele vastavad verbid, mis on eitusvormis
                                'head_id', # selles tabelis on head ID-de veeru nimi 'id'
                                'filtered.transaction_row',
                                'filtered_neg.transaction_row'
                               )

In [15]:
# filtreeritud transaction_head
create_filtered_transaction_head(cur,
                                'neg.neg_phrase_matches',
                                'head_id',
                                'filtered.transaction_head',
                                'filtered_neg.transaction_head')

### Filtreeritud transaktsioonid ilma eitusteta

In [16]:
# transaktsioonide ID-de tabeli loomine, mis vastavad transaktsioonidele, mis pole eituste hulgas

create_filtered_head_id_tbl(cur,
                           'filtered.transaction_head', # esimesest filtreeritud andmebaasist head ID-d
                           'id',
                           'filtered_neg.transaction_head', # ainult eitusi sisaldavast filtreeritud andmebaasist head ID-d
                           'id',
                           'filtered_no_neg.filtered_head_ids')

In [19]:
# filtreeritud transaction_row
create_filtered_transaction_row(cur,
                                'filtered_no_neg.filtered_head_ids',
                                'head_id',
                                'filtered.transaction_row',
                                'filtered_no_neg.transaction_row'
                               )

In [20]:
# filtreeritud transaction_head
create_filtered_transaction_head(cur,
                                'filtered_no_neg.filtered_head_ids',
                                'head_id',
                                'filtered.transaction_head',
                                'filtered_no_neg.transaction_head')

In [21]:
con.close()

NB! Andmebaasi *v32_data_filtered_no_neg.db* tabelitesse *transaction_head* ja *transaction_row* on mõned eitust sisaldavad transaktsioonid alles jäänud. Täpne põhjus on selgumisel, aga näib, et paljudel juhtudel puudub *transaction_head* tabelis olevatel neg verbidel fraasisisu *transaction_row* tabelis (ehk need verbid on üksi, võimalik, et varasema deprelite filtreerimise tagajärjel).